In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers accelerate

# ── Step 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 3: Load model (A100 40 GB — no quantisation needed) ─────────────────
import json, re, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-30B-A3B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (≈2 min on A100)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")


In [ ]:
scoring_rubric = {
  "meta": {
    "source_document": "Criteria for model.pdf",
    "generated_on": "2026-02-05",
    "notes": "Structured checklist points extracted from the provided criteria document."
  },
  "General": {
    "common_characteristics_of_good_applications": [
      "Clear research vision and career trajectory",
      "Distinctive ideas with an independent direction",
      "Well-justified training plan",
      "Strong mentorship and institutional support"
    ],
    "tell_us_why_you_need_this_award": [
      "Explain why this award is the right next step in your career (not just another project grant)",
      "What difference would securing an award make to your career, and why now?",
      "How will an award build on your existing experience and what new skills will it enable you to gain?",
      "How will it develop your research partnerships within the NHS, public health or social care, charity organisations, industry or other organisations?"
    ],
    "Applicant": [
      {
        "name": "Relevant Research Experience and Training",
        "definition": "Demonstrates prior research experience and/or training that prepares the applicant to successfully undertake the proposed award in applied health or social care research.",
        "signals": [
          "Mentions of previous projects, research methods, or techniques relevant to the proposed study",
          "Evidence of formal training, courses, or certifications",
          "Alignment between prior experience and skills required for the proposed project"
        ]
      },
      {
        "name": "Impactful Research Outputs and Outcomes",
        "definition": "Evidence of meaningful research outputs or outcomes appropriate to the applicant’s career stage, demonstrating contribution to the field.",
        "signals": [
          "Number and quality of publications, reports, or presentations",
          "Citation metrics or other indicators of impact (where mentioned)",
          "Awards, grants, or recognitions linked to prior research",
          "Description of practical or societal impact of previous work"
        ]
      },
      {
        "name": "Trajectory Towards Research Leadership",
        "definition": "Demonstrates potential and progression towards becoming a future leader in health or social care research.",
        "signals": [
          "Statements of career ambitions and long-term goals",
          "Evidence of increasing responsibility in previous roles (e.g., leading projects, mentoring)",
          "Participation in leadership or professional development programs",
          "Recognition by peers or institutions suggesting leadership potential"
        ]
      },
      {
        "name": "Commitment to Practitioner-Academic Career (if applicable)",
        "definition": "For applicants including a clinical or practice element, shows a clear commitment to combining research with professional practice.",
        "signals": [
          "Explicit statement of practitioner-academic career intentions",
          "Involvement in clinical or practice-based research",
          "Evidence of balancing practice responsibilities with research activities",
          "Participation in relevant professional networks or qualifications"
        ]
      }
    ]
  },
  "Proposed research": [
    {
      "name": "Quality of the Plain English Summary",
      "definition": "Clear, simple description of the research that a non-specialist audience can understand; explains problem, importance, plan, beneficiaries, and real-world use; avoids jargon; aligns with the detailed proposal.",
      "signals": [
        "Readability scores (e.g., Flesch-Kincaid)",
        "Low density of unexplained jargon or technical terms",
        "Presence of problem statement, objectives, methods, beneficiaries, and real-world impact",
        "Alignment with detailed proposal content",
        "Sentence structure and coherence for non-specialist understanding",
        "Emphasis on societal or patient relevance"
      ]
    },
    {
      "name": "Evidence Review Quality & Relevance",
      "definition": "How comprehensively and critically the application reviews existing literature and evidence.",
      "signals": [
        "Number and relevance of references cited",
        "Presence of critical analysis vs. mere summary",
        "Identification of knowledge gaps",
        "Coherence and logical flow"
      ]
    },
    {
      "name": "Research Scope & Award Appropriateness",
      "definition": "Whether the project scope matches the scale and expectations of the proposed award (doctoral, postdoctoral, etc.).",
      "signals": [
        "Alignment of aims with award level",
        "Feasibility indicators (time, complexity, tasks)",
        "Proportion of work relative to award expectations"
      ]
    },
    {
      "name": "Research Design & Methodological Rigor",
      "definition": "Clarity, appropriateness, and robustness of study design to answer the research questions.",
      "signals": [
        "Explicit description of methods",
        "Logical alignment between aims and methods",
        "Discussion of controls, sample size, or methodology justification",
        "Structured methodological sections"
      ]
    },
    {
      "name": "Potential Impact on Patients, Public, and Health Services",
      "definition": "Likely benefits or influence of the research on patients, carers, or health and care systems.",
      "signals": [
        "Mentions of direct or indirect patient/public benefits",
        "Alignment with health priorities or societal needs",
        "Clarity and specificity of expected outcomes"
      ]
    },
    {
      "name": "Engagement with People & Communities",
      "definition": "Evidence of meaningful involvement of patients, carers, or communities.",
      "signals": [
        "Use of co-design, participatory methods, or consultations",
        "References to ethical considerations and feedback mechanisms",
        "Description of involvement at multiple project stages"
      ]
    },
    {
      "name": "Justification of Resources & Award Costs",
      "definition": "Appropriateness and clarity of requested resources for the planned work.",
      "signals": [
        "Explicit links between tasks and resources requested",
        "Budget proportionality to project scope",
        "Clear rationale for funding level"
      ]
    },
    {
      "name": "Quality of Previous Research (if applicable)",
      "definition": "Prior relevant research experience demonstrating capability to deliver the proposed project.",
      "signals": [
        "Number and relevance of publications or outputs",
        "Evidence of prior successful projects",
        "Skills alignment with proposed work"
      ]
    },
    {
      "name": "Inclusive Research Design",
      "definition": "Consideration of inclusivity throughout the research lifecycle, including participant diversity and accessibility.",
      "signals": [
        "Explicit strategies for diverse recruitment",
        "Adaptive methods or accessibility considerations",
        "Equity and participation indicators"
      ]
    }
  ],
  "Training and development": [
    {
      "name": "Quality and Relevance of Proposed Training",
      "definition": "How well-structured, clear, and relevant the training plan is in relation to the project’s goals and the applicant’s research needs.",
      "signals": [
        "Completeness and clarity of training description",
        "Alignment of training topics with project methods or techniques",
        "Structured timeline or sequencing of training activities",
        "Explicit links between training content and skill acquisition"
      ]
    },
    {
      "name": "Suitability to Personal Development and Project Needs",
      "definition": "How well the proposed training addresses the applicant’s current skills gaps and supports successful project delivery.",
      "signals": [
        "Identification of personal skill gaps",
        "Clear mapping between training activities and project tasks",
        "Evidence that proposed training is targeted rather than generic",
        "Mention of mentorship, workshops, or formal courses relevant to development needs"
      ]
    },
    {
      "name": "Support for Career Aspirations",
      "definition": "The extent to which the training plan, and any professional practice components, contribute to the applicant’s long-term career goals.",
      "signals": [
        "Explicit statement of career goals",
        "Connections between training activities and future career trajectory",
        "Indications of skill transferability beyond the current project",
        "Reference to professional practice or experiential learning where applicable"
      ]
    }
  ],
  "Sites and support": [
    {
      "name": "Track Record of the Contracting Organisation (and Partners)",
      "definition": "Demonstrated expertise, experience, and research achievements of the organisation(s) in the proposed research area.",
      "signals": [
        "Publication or project history in relevant research fields",
        "Previous funding success or awards in similar domains",
        "Mention of prior collaborations or partnerships",
        "Highlighted achievements or notable outputs"
      ]
    },
    {
      "name": "Suitability of Supervisory Support and Mentorship Arrangements",
      "definition": "Quality, clarity, and appropriateness of supervision and mentoring for supporting the applicant’s research and career development.",
      "signals": [
        "Explicit description of supervisor roles and responsibilities",
        "Evidence of mentorship plans, meetings, or training support",
        "Alignment between supervisor expertise and project needs",
        "Presence of structured supervision framework (frequency, format, accountability)"
      ]
    },
    {
      "name": "Appropriateness of the Contracting Organisation to Support the Award and Career Aspirations",
      "definition": "How well the organisation (and any partners) can provide the infrastructure, resources, and environment necessary to deliver the project and support the applicant’s career growth.",
      "signals": [
        "Description of facilities, infrastructure, or resources provided",
        "Links between organisational support and project success",
        "Alignment of organisational focus with applicant’s career trajectory",
        "Evidence of tailored support for award type (doctoral, postdoctoral)"
      ]
    },
    {
      "name": "Commitment to Inclusive and Supportive Research Culture",
      "definition": "Evidence that the organisation fosters inclusivity, research integrity, and a supportive environment for all researchers.",
      "signals": [
        "Mentions of policies or practices promoting diversity, equity, and inclusion",
        "Evidence of inclusive recruitment, mentoring, or support initiatives",
        "References to adherence to research integrity principles",
        "Statements on organisational culture, training, or initiatives supporting underrepresented groups"
      ]
    }
  ],
  "Working with people and communities": [
    {
      "name": "Integration of Lived Experience in Research Design",
      "definition": "How input from patients, carers, or people with relevant lived experience has shaped the research questions, design, and methods.",
      "signals": [
        "Mentions of consultation or co-design activities",
        "Specific examples of changes made to study design based on lived experience",
        "References to patient or community advisory groups"
      ]
    },
    {
      "name": "Diversity and Inclusivity of the Research Team",
      "definition": "Extent to which the research team reflects diversity in terms of skills, experience, and demographic representation.",
      "signals": [
        "Explicit mentions of team composition across gender, ethnicity, age, or other relevant characteristics",
        "Statements about inclusion in hiring or team-building",
        "Evidence of equitable task allocation and collaboration"
      ]
    },
    {
      "name": "Commitment to Maximising Public and Patient Benefit",
      "definition": "Clarity and strength of the plan to ensure research outcomes meaningfully benefit patients, carers, and the public.",
      "signals": [
        "Mentions of dissemination, knowledge translation, or implementation plans",
        "Clear links between study objectives and anticipated benefits",
        "Emphasis on actionable outcomes or improvements in health/care"
      ]
    },
    {
      "name": "Representation of Seldom Heard or Underserved Groups",
      "definition": "Degree to which participants and PPI contributors represent groups historically underrepresented in research, including those facing health inequalities.",
      "signals": [
        "Mention of specific demographic groups (age, race, ethnicity, disability, gender, sexual orientation, religion/belief, location, socioeconomic status)",
        "Evidence of targeted recruitment strategies",
        "Discussion of barriers to participation and mitigation strategies"
      ]
    },
    {
      "name": "Inclusivity Across the Research Lifecycle",
      "definition": "Demonstration that inclusion is considered at all stages of the project: planning, recruitment, conduct, analysis, and dissemination.",
      "signals": [
        "References to inclusion in study design, participant selection, and analysis",
        "Consideration of accessibility and equity in all project stages",
        "Documented plans for ongoing evaluation and adaptation"
      ]
    },
    {
      "name": "Costed and Resourced Plan for Research Inclusion",
      "definition": "Evidence that research inclusion has been properly budgeted and resourced within the proposal.",
      "signals": [
        "Explicit budget line(s) for inclusion activities (PPI, outreach, recruitment support)",
        "Mention of staff time, materials, or training dedicated to inclusion",
        "Alignment between inclusion plan and allocated resources"
      ]
    }
  ],
  "Application Form": [
    {
      "name": "Use of Formatting, Headings, and Subheadings",
      "definition": "Effective use of text formatting to make the application clear, navigable, and easy to read.",
      "signals": [
        "Presence and consistency of headings and subheadings",
        "Use of bold or emphasis for key terms",
        "Structured hierarchy of sections (logical nesting of headings)",
        "Patterns detected via text parsing for formatting markers"
      ]
    },
    {
      "name": "Adherence to Word/Character Limits",
      "definition": "Each section stays within specified word or character limits, demonstrating concise writing.",
      "signals": [
        "Character/word count per section compared to allowed limits",
        "Detection of over-length sections",
        "Indicators of conciseness (average sentence/paragraph length)"
      ]
    },
    {
      "name": "Limited Duplication and Repetition",
      "definition": "Avoidance of unnecessary repetition within or across sections.",
      "signals": [
        "Semantic similarity measures between sentences/paragraphs",
        "Detection of repeated phrases or ideas",
        "Novelty score for each section (unique content vs. overlap)"
      ]
    },
    {
      "name": "Logical Flow and Coherence Across Sections",
      "definition": "The application progresses logically, with a clear thread linking the research question, objectives, and work packages.",
      "signals": [
        "Semantic coherence between sections (using embeddings or similarity metrics)",
        "Explicit linking of objectives to methods and work packages",
        "Presence of transitional phrases or linking statements",
        "Detectable narrative thread anchored on research aims"
      ]
    }
  ]
}

In [ ]:
# ── Step 4: Configure paths and load SCORING_READY ───────────────────────────
# Upload your project folder to Drive first, then set the path below.
DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_grant_coursework")
JSON_DIR   = DRIVE_ROOT / "Data" / "json_data"

json_files = sorted(JSON_DIR.glob("*.json"))
print("Available files:")
for i, f in enumerate(json_files):
    print(f"  [{i}] {f.name}")

# ── Pick the file to score (edit here) ───────────────────────────────────────
TARGET_JSON = json_files[0]   # or: JSON_DIR / "IC00001_DF_Doctoral.json"
print(f"\nTarget: {TARGET_JSON.name}")

with open(TARGET_JSON, encoding="utf-8") as f:
    _raw = json.load(f)

sr = _raw.get("SCORING_READY", {})
print(f"award_type : {sr['award_meta']['award_type']}")
print(f"title      : {sr['award_meta']['application_title'][:80]}")
print(f"SR size    : {len(json.dumps(sr)) // 1000} KB")


def _text(payload) -> str:
    return (payload or {}).get("text", "") or ""


CATEGORY_EVIDENCE = {
    "Applicant": {
        "cv_and_experience"       : _text(sr["applicant"]["evidence"]),
        "publication_mentions"    : sr["applicant"]["signals"]["publication_mentions"],
        "grant_mentions"          : sr["applicant"]["signals"]["grant_mentions"],
        "leadership_mentions"     : sr["applicant"]["signals"]["leadership_mentions"],
        "lead_applicant_info"     : sr["applicant"]["lead_applicant"],
        "supporting_team"         : sr["applicant"]["supporting_team"],
    },
    "Proposed Research": {
        "plain_english_summary"   : _text(sr["research"]["plain_english_summary"]),
        "scientific_abstract"     : _text(sr["research"]["scientific_abstract"]),
        "background_and_rationale": _text(sr["research"]["background_and_rationale"]),
        "aims"                    : _text(sr["research"]["aims"]),
        "methods"                 : _text(sr["research"]["methods"]),
        "impact_and_dissemination": _text(sr["research"]["impact_and_dissemination"]),
        "inclusive_design"        : _text(sr["research"]["inclusive_design"]),
    },
    "Training and Development": {
        "overall_plan"            : _text(sr["training"]["overall_plan"]),
        "gap_analysis"            : _text(sr["training"]["gap_analysis"]),
        "training_plan"           : _text(sr["training"]["training_plan"]),
        "leadership_development"  : _text(sr["training"]["leadership_development"]),
        "training_items"          : sr["training"]["training_items"],
    },
    "Sites and Support": {
        "supervision_team"        : sr["support"]["supervision_and_mentorship"],
        "support_text"            : _text(sr["support"]["support_text"]),
        "host_support_statement"  : _text(sr["support"]["host_support_statement"]),
        "meeting_frequency"       : sr["support"]["meeting_frequency_mentions"],
        "research_environment"    : _text(sr["support"]["research_environment"]),
    },
    "Working with People": {
        "overall_ppi"             : _text(sr["ppi_and_inclusion"]["overall_ppi"]),
        "prior_involvement"       : _text(sr["ppi_and_inclusion"]["prior_involvement"]),
        "planned_involvement"     : _text(sr["ppi_and_inclusion"]["planned_involvement"]),
        "underserved_groups"      : sr["ppi_and_inclusion"]["underserved_groups_mentioned"],
        "costed_inclusion"        : sr["ppi_and_inclusion"]["costed_inclusion_present"],
    },
    "Application Form": {
        "word_count_by_section"   : sr["form_quality"]["word_count_by_section"],
        "over_limit_sections"     : sr["form_quality"]["over_limit_sections"],
        "cross_section_similarity": sr["form_quality"]["cross_section_similarity"],
        "detected_heading_count"  : sr["form_quality"]["detected_heading_count"],
    },
}

print("\nEvidence size per category:")
for cat, fields in CATEGORY_EVIDENCE.items():
    size = len(json.dumps(fields, ensure_ascii=False))
    print(f"  {cat:<30} {size // 1000:>3} KB  (~{size // 4 // 1000}K tokens)")


In [ ]:
# ── Step 5: Score each criteria category ─────────────────────────────────────

SYSTEM_PROMPT = "You are an expert reviewer for NIHR grant applications.\nYou will be given:\n1. CRITERIA: a list of scoring criteria with definitions and signals\n2. EVIDENCE: extracted content from the grant application relevant to those criteria\n\nFor each criterion, output a JSON object with:\n- \"score\": integer 1-10\n- \"evidence_quote\": a short direct quote from the evidence (max 60 words), or \"\" if absent\n- \"reasoning\": your justification (max 80 words)\n\nReturn ONLY a valid JSON object like:\n{\n  \"Criterion Name\": {\"score\": 7, \"evidence_quote\": \"...\", \"reasoning\": \"...\"},\n  ...\n}\nDo not include any text outside the JSON."


def run_scoring(category_name: str, criteria_list: list, evidence: dict) -> dict:
    user_prompt = (
        f"CATEGORY: {category_name}\n\n"
        f"CRITERIA:\n{json.dumps(criteria_list, ensure_ascii=False, indent=2)}\n\n"
        f"EVIDENCE FROM APPLICATION:\n{json.dumps(evidence, ensure_ascii=False, indent=2)}\n\n"
        "Score each criterion above. Return only valid JSON."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # Strip <think> block, keep only final answer
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    final_text = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip()

    return _extract_json(final_text)


def _extract_json(text: str) -> dict:
    """Try multiple strategies to extract JSON from model output."""
    # 1. markdown code block: ```json { ... } ```
    md = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if md:
        try:
            return json.loads(md.group(1))
        except json.JSONDecodeError:
            pass

    # 2. bare JSON object (greedy from first { to last })
    start = text.find('{')
    end   = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass

    # 3. fallback: save raw for inspection
    return {"_raw_output": text, "_parse_error": True}


CRITERIA_MAP = {
    "Applicant"               : scoring_rubric["General"]["Applicant"],
    "Proposed Research"       : scoring_rubric["Proposed research"],
    "Training and Development": scoring_rubric["Training and development"],
    "Sites and Support"       : scoring_rubric["Sites and support"],
    "Working with People"     : scoring_rubric["Working with people and communities"],
    "Application Form"        : scoring_rubric["Application Form"],
}

all_scores = {
    "_meta": {
        "file"      : TARGET_JSON.name,
        "award_type": sr["award_meta"]["award_type"],
        "title"     : sr["award_meta"]["application_title"],
    }
}

for category, criteria in CRITERIA_MAP.items():
    print(f"\n{'='*60}")
    print(f"Scoring: {category}  ({len(criteria)} criteria) ...")
    result = run_scoring(category, criteria, CATEGORY_EVIDENCE[category])
    all_scores[category] = result

    if result.get("_parse_error"):
        print("  [WARNING] JSON parse failed — raw output saved")
    else:
        for crit_name, scores in result.items():
            print(f"  {crit_name[:52]:<52} {scores.get('score', '?'):>2}/10")

# Save to Drive
out_dir = DRIVE_ROOT / "scores"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / (TARGET_JSON.stem + "_scores.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_scores, f, ensure_ascii=False, indent=2)

print(f"\n✓ Scores saved to: {out_path}")
